In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [2]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print(model_name)
print("hidden_size:", model.config.hidden_size)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbert-base-uncased
hidden_size: 768


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
def masked_max_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand_as(last_hidden_state).bool()
    masked_hidden = last_hidden_state.masked_fill(~mask, -1e9)
    pooled = masked_hidden.max(dim=1).values
    pooled = torch.nn.functional.normalize(pooled, p=2, dim=-1)
    return pooled


def encode_texts(texts, batch_size=128, max_length=128):
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i + batch_size]
            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = model(**enc)
            pooled = masked_max_pool(outputs.last_hidden_state, enc["attention_mask"])
            all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0)


test_hidden = torch.randn(2, 4, 8)
test_mask = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0]])
test_pooled = masked_max_pool(test_hidden, test_mask)
print("test pooled shape:", tuple(test_pooled.shape))

test pooled shape: (2, 8)


In [5]:
batch_size = 128
threshold = 0.85

emb1 = encode_texts(sent1, batch_size=batch_size)
emb2 = encode_texts(sent2, batch_size=batch_size)

cosine_scores = (emb1 * emb2).sum(dim=-1).numpy()
y_pred = (cosine_scores >= threshold).astype(int)

print("emb1:", tuple(emb1.shape))
print("emb2:", tuple(emb2.shape))
print("done")

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

emb1: (408, 768)
emb2: (408, 768)
done


In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1, "threshold": threshold})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))

{'accuracy': 0.6838235294117647, 'f1': 0.8122270742358079, 'threshold': 0.85}
                precision    recall  f1-score   support

not_paraphrase       0.00      0.00      0.00       129
    paraphrase       0.68      1.00      0.81       279

      accuracy                           0.68       408
     macro avg       0.34      0.50      0.41       408
  weighted avg       0.47      0.68      0.56       408



/Users/jinjinzhao/Documents/work_projects/tablevault_experiments/tablevault_experiments/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/jinjinzhao/Documents/work_projects/tablevault_experiments/tablevault_experiments/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/jinjinzhao/Documents/work_projects/tablevault_experiments/tablevault_experiments/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Prec

In [7]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("cosine:", float(cosine_scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("cosine:", float(cosine_scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
cosine: 0.9697094559669495
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
cosine: 0.9531527161598206
true: 0 pred: 1
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
cosine: 0.9761152267456055
true: 0 pred: 1
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will 

In [8]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(device),
    "pooling": "masked_max",
    "threshold": float(threshold),
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
}
summary

{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'distilbert-base-uncased',
 'device': 'mps',
 'pooling': 'masked_max',
 'threshold': 0.85,
 'num_examples': 408,
 'accuracy': 0.6838235294117647,
 'f1': 0.8122270742358079}